In [20]:
import httpx
url = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
params = {"$limit": 5, "$select": "nombre_entidad,id_contrato,valor_del_contrato", "$order": "id_contrato"}
respuesta = httpx.get(url, params=params, timeout=60)

print(respuesta.status_code)
print(respuesta.json())

200
[{'nombre_entidad': 'SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.', 'id_contrato': 'CO1.PCCNTR.10000001', 'valor_del_contrato': '2743872'}, {'nombre_entidad': 'ICBF REGIONAL CAUCA', 'id_contrato': 'CO1.PCCNTR.10000002', 'valor_del_contrato': '2675520'}, {'nombre_entidad': 'INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN', 'id_contrato': 'CO1.PCCNTR.10000003', 'valor_del_contrato': '20115161'}, {'nombre_entidad': 'DEPARTAMENTO DEL VALLE DEL CAUCA-SECRETARIA DE GESTION DEL RIESGO DE DESASTRES', 'id_contrato': 'CO1.PCCNTR.10000004', 'valor_del_contrato': '10000000'}, {'nombre_entidad': 'SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.', 'id_contrato': 'CO1.PCCNTR.10000005', 'valor_del_contrato': '7765376'}]


La respuesta es una lista de diccionarios
El valor del contrato está en comillas, debe pasarse a número para poder realizar operaciones

In [21]:
import polars as pl
df = pl.DataFrame(respuesta.json())



valor_del_contrato tiene tipo str, lo que confirma que la API (SODA 2) entrega el dato en tipo texto

In [22]:
tabla = df.with_columns(pl.col("valor_del_contrato").cast(pl.Float64))
print(tabla )  
print(tabla.schema)

tabla["valor_del_contrato"].sum()   

shape: (5, 3)
┌─────────────────────────────────┬─────────────────────┬────────────────────┐
│ nombre_entidad                  ┆ id_contrato         ┆ valor_del_contrato │
│ ---                             ┆ ---                 ┆ ---                │
│ str                             ┆ str                 ┆ f64                │
╞═════════════════════════════════╪═════════════════════╪════════════════════╡
│ SUBRED INTEGRADA DE SERVICIOS … ┆ CO1.PCCNTR.10000001 ┆ 2.743872e6         │
│ ICBF REGIONAL CAUCA             ┆ CO1.PCCNTR.10000002 ┆ 2.67552e6          │
│ INSTITUTO DE DEPORTES Y RECREA… ┆ CO1.PCCNTR.10000003 ┆ 2.0115161e7        │
│ DEPARTAMENTO DEL VALLE DEL CAU… ┆ CO1.PCCNTR.10000004 ┆ 1e7                │
│ SUBRED INTEGRADA DE SERVICIOS … ┆ CO1.PCCNTR.10000005 ┆ 7.765376e6         │
└─────────────────────────────────┴─────────────────────┴────────────────────┘
Schema({'nombre_entidad': String, 'id_contrato': String, 'valor_del_contrato': Float64})


43299929.0

Se utiliza Float64 pues acepta cualquier valor tenga decimales o no. probar en muestra más grande. df tiene valor_del_contrato en str, mientras que tabla en float64